In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
        .appName('DataTransformations')\
        .master('local[*]')\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/23 14:27:36 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.190 instead (on interface en0)
26/03/23 14:27:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 14:27:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
data = [
    (1, "Alice", "Laptop", 1200, "US"),
    (2, "Bob", "Phone", 800, "US"),
    (3, "Alice", "Mouse", 25, "UK"),
    (4, "David", "Laptop", 1300, "US"),
    (5, "Eve", "Keyboard", 75, "IN"),
    (6, "Frank", "Phone", 900, "US"),
    (7, "Alice", "Laptop", 1150, "UK")
]

columns = ["user_id", "user_name", "product", "amount", "country"]


orders_df = spark.createDataFrame(data,columns)

In [10]:
from pyspark.sql.functions import col, when, lower



In [8]:
orders_df.filter(col('country')=='US')\
         .select("user_name","product","amount")\
         .withColumn('high_value',
                    when(col('amount')>1000,'yes').otherwise('No')
                    ).explain()

== Physical Plan ==
*(1) Project [user_name#1, product#2, amount#3L, CASE WHEN (amount#3L > 1000) THEN yes ELSE No END AS high_value#20]
+- *(1) Filter (isnotnull(country#4) AND (country#4 = US))
   +- *(1) Scan ExistingRDD[user_id#0L,user_name#1,product#2,amount#3L,country#4]




In [12]:
data = [
    (1, "Alice", "Laptop", 1200, "US", "completed"),
    (2, "Bob", "Phone", 800, "US", "completed"),
    (3, "Alice", "Mouse", 25, "UK", "returned"),
    (4, "David", "Laptop", 1300, "US", "completed"),
    (5, "Eve", "Keyboard", 75, "IN", "pending"),
    (6, "Frank", "Phone", 900, "US", "completed"),
    (7, "Alice", "Laptop", 1150, "UK", "completed"),
    (8, "Bob", "Monitor", 300, "US", "pending")
]

columns = ["user_id", "user_name", "product", "amount", "country", "status"]
orders_df_new = spark.createDataFrame(data,columns)

In [17]:
orders_df_new.withColumn('new_amt',when(col('amount')>=1000,'distcount')).show()

+-------+---------+--------+------+-------+---------+---------+
|user_id|user_name| product|amount|country|   status|  new_amt|
+-------+---------+--------+------+-------+---------+---------+
|      1|    Alice|  Laptop|  1200|     US|completed|distcount|
|      2|      Bob|   Phone|   800|     US|completed|     NULL|
|      3|    Alice|   Mouse|    25|     UK| returned|     NULL|
|      4|    David|  Laptop|  1300|     US|completed|distcount|
|      5|      Eve|Keyboard|    75|     IN|  pending|     NULL|
|      6|    Frank|   Phone|   900|     US|completed|     NULL|
|      7|    Alice|  Laptop|  1150|     UK|completed|distcount|
|      8|      Bob| Monitor|   300|     US|  pending|     NULL|
+-------+---------+--------+------+-------+---------+---------+



In [21]:
orders_df_new \
    .select("user_name", "product", "status", "amount", "country") \
    .filter(col("status") == "completed") \
    .withColumn(
        "discount_flag",
        when(col("amount") >= 1000, "discount").otherwise("regular")
    ) \
    .withColumn("country_code", lower(col("country"))) \
    .select("user_name", "product", "amount", "country_code", "discount_flag") \
    .show()

+---------+-------+------+------------+-------------+
|user_name|product|amount|country_code|discount_flag|
+---------+-------+------+------------+-------------+
|    Alice| Laptop|  1200|          us|     discount|
|      Bob|  Phone|   800|          us|      regular|
|    David| Laptop|  1300|          us|     discount|
|    Frank|  Phone|   900|          us|      regular|
|    Alice| Laptop|  1150|          uk|     discount|
+---------+-------+------+------------+-------------+



In [22]:
from pyspark.sql.functions import col, when, lower, upper, lit

data = [
    (1, "Alice", "Laptop", 1200, "US", "completed"),
    (2, "Bob", "Phone", 800, "US", "completed"),
    (3, "Alice", "Mouse", 25, "UK", "returned"),
    (4, "David", "Laptop", 1300, "US", "completed"),
    (5, "Eve", "Keyboard", 75, "IN", "pending"),
    (6, "Frank", "Phone", 900, "US", "completed"),
    (7, "Alice", "Laptop", 1150, "UK", "completed"),
    (8, "Bob", "Monitor", 300, "US", "pending"),
    (9, "George", "Tablet", 600, "CA", "completed")
]

columns = ["user_id", "user_name", "product", "amount", "country", "status"]

orders_df = spark.createDataFrame(data, columns)

In [23]:
orders_df.show()

+-------+---------+--------+------+-------+---------+
|user_id|user_name| product|amount|country|   status|
+-------+---------+--------+------+-------+---------+
|      1|    Alice|  Laptop|  1200|     US|completed|
|      2|      Bob|   Phone|   800|     US|completed|
|      3|    Alice|   Mouse|    25|     UK| returned|
|      4|    David|  Laptop|  1300|     US|completed|
|      5|      Eve|Keyboard|    75|     IN|  pending|
|      6|    Frank|   Phone|   900|     US|completed|
|      7|    Alice|  Laptop|  1150|     UK|completed|
|      8|      Bob| Monitor|   300|     US|  pending|
|      9|   George|  Tablet|   600|     CA|completed|
+-------+---------+--------+------+-------+---------+



In [25]:
orders_df.filter(col('status')!='returned')\
         .withColumn('user_name_upper', upper(col('user_name')))\
         .withColumn('user_band',
             when(col('amount')>=1000,"high")\
            .when((col('amount')>=500) & (col('amount')<1000), "medium")\
            .otherwise("low")
           )\
         .withColumn('is_us_order',
                     when(col('country')=='US',"yes").otherwise('no')
                    )\
         .select('user_id','user_name_upper','product','amount','user_band','is_us_order')\
         .show()

+-------+---------------+--------+------+---------+-----------+
|user_id|user_name_upper| product|amount|user_band|is_us_order|
+-------+---------------+--------+------+---------+-----------+
|      1|          ALICE|  Laptop|  1200|     high|        yes|
|      2|            BOB|   Phone|   800|   medium|        yes|
|      4|          DAVID|  Laptop|  1300|     high|        yes|
|      5|            EVE|Keyboard|    75|      low|         no|
|      6|          FRANK|   Phone|   900|   medium|        yes|
|      7|          ALICE|  Laptop|  1150|     high|         no|
|      8|            BOB| Monitor|   300|      low|        yes|
|      9|         GEORGE|  Tablet|   600|   medium|         no|
+-------+---------------+--------+------+---------+-----------+



In [26]:
orders_df.show()

+-------+---------+--------+------+-------+---------+
|user_id|user_name| product|amount|country|   status|
+-------+---------+--------+------+-------+---------+
|      1|    Alice|  Laptop|  1200|     US|completed|
|      2|      Bob|   Phone|   800|     US|completed|
|      3|    Alice|   Mouse|    25|     UK| returned|
|      4|    David|  Laptop|  1300|     US|completed|
|      5|      Eve|Keyboard|    75|     IN|  pending|
|      6|    Frank|   Phone|   900|     US|completed|
|      7|    Alice|  Laptop|  1150|     UK|completed|
|      8|      Bob| Monitor|   300|     US|  pending|
|      9|   George|  Tablet|   600|     CA|completed|
+-------+---------+--------+------+-------+---------+



In [31]:
orders_df.withColumn(
                    'normalized_amount',
                     when(col('country')=='US',col('amount'))\
                    .otherwise(col('amount')*0.8)
                    )\
        .filter(col('normalized_amount')>=500)\
        .withColumn('user_name_upper', upper(col('user_name')))\
        .withColumn('order_band',
             when(col('amount')>=1000,"high")\
            .when((col('amount')>=500) & (col('amount')<1000), "medium")\
            .otherwise("low")
           )\
        .withColumn('is_us_order',
                     when(col('country')=='US',"yes").otherwise('no')
                    )\
        .select('user_id','user_name_upper','product','amount','order_band','is_us_order')\
         .show()

+-------+---------------+-------+------+----------+-----------+
|user_id|user_name_upper|product|amount|order_band|is_us_order|
+-------+---------------+-------+------+----------+-----------+
|      1|          ALICE| Laptop|  1200|      high|        yes|
|      2|            BOB|  Phone|   800|    medium|        yes|
|      4|          DAVID| Laptop|  1300|      high|        yes|
|      6|          FRANK|  Phone|   900|    medium|        yes|
|      7|          ALICE| Laptop|  1150|      high|         no|
+-------+---------------+-------+------+----------+-----------+



In [44]:
data = [
    (101, "alice", "laptop", 1200, "US", "completed", "electronics"),
    (102, "bob", "phone", 800, "US", "completed", "electronics"),
    (103, "alice", "mouse", 25, "UK", "returned", "accessories"),
    (104, "david", "laptop", 1300, "US", "completed", "electronics"),
    (105, "eve", "keyboard", 75, "IN", "pending", "accessories"),
    (106, "frank", "phone", 900, "US", "completed", "electronics"),
    (107, "alice", "laptop", 1150, "UK", "completed", "electronics"),
    (108, "bob", "monitor", 300, "US", "pending", "electronics"),
    (109, "george", "tablet", 600, "CA", "completed", "electronics"),
    (110, "helen", "chair", 200, "US", "completed", "furniture"),
    (111, "ian", "desk", 450, "CA", "pending", "furniture"),
    (112, "jane", "mouse", 40, "US", "completed", "accessories")
]

columns = ["order_id", "user_name", "product", "amount", "country", "status", "category"]

orders_df = spark.createDataFrame(data, columns)

# QUESTIONS

### a) Basic

- Keep only completed orders and select:
  - order_id  
  - user_name  
  - product  
  - amount  

- Create a new column `amount_flag`:
  - "high" if amount >= 1000  
  - "low" otherwise  

In [45]:
orders_df.filter(col('status')=='completed')\
         .withColumn('amount_flag', 
                      when(col('amount')>=1000,'high').otherwise('low'))\
         .select(
             'order_id',
             'user_name',
             'product',
             'amount'
         ).show()

+--------+---------+-------+------+
|order_id|user_name|product|amount|
+--------+---------+-------+------+
|     101|    alice| laptop|  1200|
|     102|      bob|  phone|   800|
|     104|    david| laptop|  1300|
|     106|    frank|  phone|   900|
|     107|    alice| laptop|  1150|
|     109|   george| tablet|   600|
|     110|    helen|  chair|   200|
|     112|     jane|  mouse|    40|
+--------+---------+-------+------+



### b) Medium
#### 1. Filter + Transform
- Keep only rows where `status != 'returned'`

Create:
- `user_name_clean` = initcap(user_name)
- `product_upper` = uppercase(product)

Final columns:
- order_id  
- user_name_clean  
- product_upper  
- status  

---

#### 2. Regional Amount + Spend Band

Create:
- `regional_amount`:
  - if country = 'US' -> amount  
  - otherwise -> amount * 0.9  

- `spend_band`:
  - high if regional_amount >= 1000  
  - medium if regional_amount >= 500  
  - low otherwise  

Final columns:
- order_id  
- country  
- amount  
- regional_amount  
- spend_band  

---

#### 3. Electronics Category Logic

Filter:
- category = 'electronics'

Create:
- `priority_order`:
  - "yes" if amount >= 900  
  - "no" otherwise  

- `country_lower` = lowercase(country)

Final columns:
- order_id  
- product  
- amount  
- priority_order  
- country_lower  

---

#### 4. Status Label

Create:
- `status_label`:
  - "done" -> completed  
  - "in_progress" -> pending  
  - "other" -> otherwise  

Final columns:
- order_id  
- status  
- status_label  

---

#### 5. Order Description

Create:
- `order_desc` = concat(user_name, product, country, separator = " | ")

Final columns:
- order_id  
- order_desc  


In [53]:
df1 = orders_df.filter(col('status') != 'returned') \
    .withColumn('user_name_clean', initcap(col('user_name'))) \
    .withColumn('product_upper', upper(col('product'))) \
    .select('order_id', 'user_name_clean', 'product_upper', 'status')

In [54]:
df2 = orders_df.withColumn(
    'regional_amount',
    when(col('country')=='US', col('amount')).otherwise(col('amount')*0.9)
).withColumn(
    'spend_band',
    when(col('regional_amount')>=1000,'high')
    .when(col('regional_amount')>=500,'medium')
    .otherwise('low')
).select('order_id','country','amount','regional_amount','spend_band')

In [55]:
df3 = orders_df.filter(col('category')=='electronics') \
    .withColumn('priority_order',
        when(col('amount')>=900,'yes').otherwise('no')
    ).withColumn('country_lower', lower(col('country'))) \
    .select('order_id','product','amount','priority_order','country_lower')

In [56]:
df4 = orders_df.withColumn(
    'status_label',
    when(col('status')=='completed','done')
    .when(col('status')=='pending','in_progress')
    .otherwise('other')
).select('order_id','status','status_label')

In [57]:
df5 = orders_df.withColumn(
    'order_desc',
    concat_ws(' | ', col('user_name'), col('product'), col('country'))
).select('order_id','order_desc')

NameError: name 'concat_ws' is not defined

### c) HARD — FAANG LEVEL

---

#### 1. Normalized Amount + Risk + Category Tag

Create:
- `normalized_amount`:
  - US -> amount  
  - CA -> amount * 0.95  
  - others -> amount * 0.85  

Filter:
- normalized_amount >= 500  

Create:
- `risk_flag`:
  - "review" if status = pending AND normalized_amount >= 700  
  - "clear" otherwise  

- `category_tag`:
  - "core" if category IN (electronics, furniture)  
  - "non_core" otherwise  

Final columns:
- order_id  
- country  
- status  
- category  
- normalized_amount  
- risk_flag  
- category_tag  

---

#### 2. User + Product Group + Amount Bucket

Filter:
- status IN (completed, pending)  

Create:
- `clean_user` = initcap(user_name)  

- `product_group`:
  - "device" -> laptop, phone, tablet, monitor  
  - "workspace" -> chair, desk  
  - "peripheral" otherwise  

- `amount_bucket`:
  - "A" if amount >= 1000  
  - "B" if amount >= 500  
  - "C" otherwise  

Final columns:
- order_id  
- clean_user  
- product  
- product_group  
- amount_bucket  

---

#### 3. Business Status + Geo Segment

Create:
- `business_status`:
  - completed AND amount >= 1000 -> "high_value_closed"  
  - completed AND amount < 1000 -> "closed"  
  - pending -> "open"  
  - returned -> "reverse_flow"  

- `geo_segment`:
  - "north_america" -> US, CA  
  - "international" otherwise  

Final columns:
- order_id  
- business_status  
- geo_segment  
- amount  

---

#### 4. Standardization + Fee Adjustment

Create:
- `user_name_std` = uppercase(user_name)  
- `category_std` = lowercase(category)  

- `status_std`:
  - completed -> COMPLETE  
  - pending -> PENDING  
  - returned -> RETURNED  

- `amount_after_fee`:
  - electronics -> amount * 0.98  
  - furniture -> amount * 0.95  
  - accessories -> amount * 0.90  

Final columns:
- order_id  
- user_name_std  
- category_std  
- status_std  
- amount_after_fee  

---

#### 5. Operational Pipeline

Filter:
- exclude returned orders  

Create:
- `domestic_flag`:
  - "domestic" -> US  
  - "international" otherwise  

- `amount_tier`:
  - "platinum" if amount >= 1200  
  - "gold" if amount >= 800  
  - "silver" otherwise  

- `ops_action`:
  - "expedite" if status = completed AND amount >= 1000  
  - "hold" if status = pending  
  - "standard" otherwise  

Final columns:
- order_id  
- domestic_flag  
- amount_tier  
- ops_action  

In [68]:
orders_df.withColumn('normalized_amount',
                    when(col('country')=='US', col('amount'))
                    .when(col('country')=='CA', col('amount')*0.95).otherwise(col('amount')*0.85)
                    )\
         .filter(col('normalized_amount')>=500)\
         .withColumn('risk_flag',
                      when((col('status')=='pending') & (col('normalized_amount')>=700),'review').otherwise('clear')
                    )\
         .withColumn('category_tag', 
                     when((col('category') =='electronics') | (col('category')=='furniture'),'core').otherwise('non_core')
         )\
         .select(
             'order_id',
             'country',
             'status',
             'category',
             'normalized_amount',
             'risk_flag',
             'category_tag'
         ).show()


+--------+-------+---------+-----------+-----------------+---------+------------+
|order_id|country|   status|   category|normalized_amount|risk_flag|category_tag|
+--------+-------+---------+-----------+-----------------+---------+------------+
|     101|     US|completed|electronics|           1200.0|    clear|        core|
|     102|     US|completed|electronics|            800.0|    clear|        core|
|     104|     US|completed|electronics|           1300.0|    clear|        core|
|     106|     US|completed|electronics|            900.0|    clear|        core|
|     107|     UK|completed|electronics|            977.5|    clear|        core|
|     109|     CA|completed|electronics|            570.0|    clear|        core|
+--------+-------+---------+-----------+-----------------+---------+------------+



In [66]:
my_list  = ['laptop','phone','tablet','monitor']
my_list2 = ['chair','desk']

orders_df.filter((col('status')=='completed') | (col('status')=='pending'))\
        .withColumn('clean_user', initcap(col('user_name')))\
        .withColumn('product_group', 
                    when(col('product').isin(my_list),'device')
                    .when(col('product').isin(my_list2),'workspace').otherwise('peripheral')
                   )\
        .withColumn('amount_bucket',
                     when(col('amount')>=1000,'A')
                    .when(col('amount')>=500,'B').otherwise('C')
                   )\
        .select(
            'order_id',
            'clean_user',
            'product',
            'product_group',
            'amount_bucket'
        ).show()

+--------+----------+--------+-------------+-------------+
|order_id|clean_user| product|product_group|amount_bucket|
+--------+----------+--------+-------------+-------------+
|     101|     Alice|  laptop|       device|            A|
|     102|       Bob|   phone|       device|            B|
|     104|     David|  laptop|       device|            A|
|     105|       Eve|keyboard|   peripheral|            C|
|     106|     Frank|   phone|       device|            B|
|     107|     Alice|  laptop|       device|            A|
|     108|       Bob| monitor|       device|            C|
|     109|    George|  tablet|       device|            B|
|     110|     Helen|   chair|    workspace|            C|
|     111|       Ian|    desk|    workspace|            C|
|     112|      Jane|   mouse|   peripheral|            C|
+--------+----------+--------+-------------+-------------+



In [69]:
orders_df \
    .withColumn(
        'business_status',
        when((col('status') == 'completed') & (col('amount') >= 1000), 'high_value_closed')
        .when((col('status') == 'completed') & (col('amount') < 1000), 'closed')
        .when(col('status') == 'pending', 'open')
        .when(col('status') == 'returned', 'reverse_flow')
    ) \
    .withColumn(
        'geo_segment',
        when(col('country').isin('US', 'CA'), 'north_america').otherwise('international')
    ) \
    .select('order_id', 'business_status', 'geo_segment', 'amount') \
    .show()

+--------+-----------------+-------------+------+
|order_id|  business_status|  geo_segment|amount|
+--------+-----------------+-------------+------+
|     101|high_value_closed|north_america|  1200|
|     102|           closed|north_america|   800|
|     103|     reverse_flow|international|    25|
|     104|high_value_closed|north_america|  1300|
|     105|             open|international|    75|
|     106|           closed|north_america|   900|
|     107|high_value_closed|international|  1150|
|     108|             open|north_america|   300|
|     109|           closed|north_america|   600|
|     110|           closed|north_america|   200|
|     111|             open|north_america|   450|
|     112|           closed|north_america|    40|
+--------+-----------------+-------------+------+



In [70]:
orders_df \
    .withColumn('user_name_std', upper(col('user_name'))) \
    .withColumn('category_std', lower(col('category'))) \
    .withColumn(
        'status_std',
        when(col('status') == 'completed', 'COMPLETE')
        .when(col('status') == 'pending', 'PENDING')
        .when(col('status') == 'returned', 'RETURNED')
    ) \
    .withColumn(
        'amount_after_fee',
        when(col('category') == 'electronics', col('amount') * 0.98)
        .when(col('category') == 'furniture', col('amount') * 0.95)
        .when(col('category') == 'accessories', col('amount') * 0.90)
    ) \
    .select('order_id', 'user_name_std', 'category_std', 'status_std', 'amount_after_fee') \
    .show()

+--------+-------------+------------+----------+----------------+
|order_id|user_name_std|category_std|status_std|amount_after_fee|
+--------+-------------+------------+----------+----------------+
|     101|        ALICE| electronics|  COMPLETE|          1176.0|
|     102|          BOB| electronics|  COMPLETE|           784.0|
|     103|        ALICE| accessories|  RETURNED|            22.5|
|     104|        DAVID| electronics|  COMPLETE|          1274.0|
|     105|          EVE| accessories|   PENDING|            67.5|
|     106|        FRANK| electronics|  COMPLETE|           882.0|
|     107|        ALICE| electronics|  COMPLETE|          1127.0|
|     108|          BOB| electronics|   PENDING|           294.0|
|     109|       GEORGE| electronics|  COMPLETE|           588.0|
|     110|        HELEN|   furniture|  COMPLETE|           190.0|
|     111|          IAN|   furniture|   PENDING|           427.5|
|     112|         JANE| accessories|  COMPLETE|            36.0|
+--------+

In [71]:
orders_df \
    .filter(col('status') != 'returned') \
    .withColumn(
        'domestic_flag',
        when(col('country') == 'US', 'domestic').otherwise('international')
    ) \
    .withColumn(
        'amount_tier',
        when(col('amount') >= 1200, 'platinum')
        .when(col('amount') >= 800, 'gold')
        .otherwise('silver')
    ) \
    .withColumn(
        'ops_action',
        when((col('status') == 'completed') & (col('amount') >= 1000), 'expedite')
        .when(col('status') == 'pending', 'hold')
        .otherwise('standard')
    ) \
    .select('order_id', 'domestic_flag', 'amount_tier', 'ops_action') \
    .show()

+--------+-------------+-----------+----------+
|order_id|domestic_flag|amount_tier|ops_action|
+--------+-------------+-----------+----------+
|     101|     domestic|   platinum|  expedite|
|     102|     domestic|       gold|  standard|
|     104|     domestic|   platinum|  expedite|
|     105|international|     silver|      hold|
|     106|     domestic|       gold|  standard|
|     107|international|       gold|  expedite|
|     108|     domestic|     silver|      hold|
|     109|international|     silver|  standard|
|     110|     domestic|     silver|  standard|
|     111|international|     silver|      hold|
|     112|     domestic|     silver|  standard|
+--------+-------------+-----------+----------+

